Clean the Income data

In [ ]:
import pandas as pd
import glob
import re
import os

def clean_census_income_auto(file_path):
    # 1. Extract the Year automatically from the filename
    match = re.search(r'20\d{2}', os.path.basename(file_path))
    year = int(match.group(0)) if match else None

    # 2. Load the file
    df = pd.read_csv(file_path)

    # 3. Find the 'Median income (dollars)' row
    median_row = df[df['Label (Grouping)'].str.contains('Median income \(dollars\)', na=False, case=False)]
    if median_row.empty: return None

    # 4. Identify the 'Households!!Estimate' columns for each state
    target_cols = [c for c in df.columns if 'Households!!Estimate' in c]

    state_data = []
    for col in target_cols:
        state_name = col.split('!!')[0]
        income_val = median_row[col].values[0]

        # 5. Clean numeric strings (remove commas, handle Census symbols like 'N' or '-')
        try:
            if pd.isna(income_val) or str(income_val).strip() in ['-', 'N', '(X)']:
                income_val = None
            else:
                clean_val = str(income_val).replace(',', '').replace('+', '').replace('$', '').strip()
                income_val = float(clean_val)
        except ValueError:
            income_val = None

        state_data.append({'State': state_name, 'Year': year, 'Median_Income': income_val})

    return pd.DataFrame(state_data)

# --- COMBINE ALL YEARS ---
# This finds every S1901 file in your folder automatically
all_income_files = glob.glob('ACSST*Y*.S1901*.csv')
all_dfs = []

for f in all_income_files:
    clean_df = clean_census_income_auto(f)
    if clean_df is not None:
        all_dfs.append(clean_df)

if all_dfs:
    # Stack all years together
    full_panel = pd.concat(all_dfs, ignore_index=True)

    # Filter for your specific 2020-2024 range
    final_panel = full_panel[full_panel['Year'].between(2020, 2024)]

    # Save the final file
    final_panel.to_csv('income_panel_2020_2024.csv', index=False)
    print("Success! Your 2020-2024 panel is ready.")

Success! Your 2020-2024 panel is ready.


<>:15: SyntaxWarning: invalid escape sequence '\('
<>:15: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_9471/4282626986.py:15: SyntaxWarning: invalid escape sequence '\('
  median_row = df[df['Label (Grouping)'].str.contains('Median income \(dollars\)', na=False, case=False)]


Clean the Poverty data

In [ ]:
def clean_census_poverty_auto(file_path):
    # 1. Extract the Year from the filename (e.g., 2021)
    match = re.search(r'20\d{2}', os.path.basename(file_path))
    year = int(match.group(0)) if match else None

    # 2. Load the file
    df = pd.read_csv(file_path)

    # 3. Locate the row for the total population poverty status
    poverty_row = df[df['Label (Grouping)'].str.contains('poverty status is determined', na=False, case=False)]

    # If the label match fails, fallback to the very first data row
    if poverty_row.empty:
        poverty_row = df.head(1)

    # 4. Identify the 'Percent below poverty level!!Estimate' columns
    target_cols = [c for c in df.columns if 'Percent below poverty level!!Estimate' in c]

    state_data = []
    for col in target_cols:
        state_name = col.split('!!')[0] # Get state name
        poverty_rate = poverty_row[col].values[0]

        # 5. Clean numeric strings (remove %, commas, and Census symbols)
        try:
            if pd.isna(poverty_rate) or str(poverty_rate).strip() in ['-', 'N', '(X)']:
                poverty_rate = None
            else:
                clean_val = str(poverty_rate).replace('%', '').replace(',', '').strip()
                poverty_rate = float(clean_val)
        except ValueError:
            poverty_rate = None

        state_data.append({'State': state_name, 'Year': year, 'Poverty_Rate_Pct': poverty_rate})

    return pd.DataFrame(state_data)

# --- COMBINE ALL YEARS ---
all_poverty_files = glob.glob('ACSST*Y*.S1701*.csv')
all_dfs = []

for f in all_poverty_files:
    clean_df = clean_census_poverty_auto(f)
    if clean_df is not None:
        all_dfs.append(clean_df)

if all_dfs:
    # Combine everything and filter for the 2020-2024 range
    poverty_panel = pd.concat(all_dfs, ignore_index=True)
    poverty_panel = poverty_panel[poverty_panel['Year'].between(2020, 2024)]

    # Save the final file
    poverty_panel.to_csv('poverty_panel_2020_2024.csv', index=False)
    print("Success! Your 2020-2024 poverty panel is ready.")

Success! Your 2020-2024 poverty panel is ready.


Clean the Education data

In [ ]:
def clean_census_edu_auto(file_path):
    # 1. Extract Year
    match = re.search(r'20\d{2}', os.path.basename(file_path))
    year = int(match.group(0)) if match else None

    # 2. Load File
    df = pd.read_csv(file_path)

    # 3. Clean labels and find the 'Bachelor's degree or higher' row for pop 25+
    df['clean_label'] = df['Label (Grouping)'].str.replace(r'\s+', ' ', regex=True).str.strip()

    try:
        # We target the row for Bachelor's degree or higher within the 25+ age group
        pop_25_idx = df[df['clean_label'] == 'Population 25 years and over'].index[0]
        target_row_idx = df[(df.index > pop_25_idx) & (df['clean_label'] == "Bachelor's degree or higher")].index[0]
        edu_row = df.iloc[[target_row_idx]]
    except:
        return None

    # 4. Pull 'Percent!!Estimate' columns for each state
    target_cols = [c for c in df.columns if '!!Percent!!Estimate' in c]

    state_data = []
    for col in target_cols:
        state_name = col.split('!!')[0]
        val = edu_row[col].values[0]

        # 5. Convert to numeric
        try:
            if pd.isna(val) or str(val).strip() in ['-', 'N', '(X)']:
                val = None
            else:
                val = float(str(val).replace(',', '').replace('%', '').strip())
        except ValueError:
            val = None

        state_data.append({'State': state_name, 'Year': year, 'Pct_Bachelors_Higher': val})

    return pd.DataFrame(state_data)

# Combine and save
edu_files = glob.glob('ACSST*Y*.S1501*.csv')
edu_panel = pd.concat([clean_census_edu_auto(f) for f in edu_files if clean_census_edu_auto(f) is not None])
edu_panel.to_csv('education_panel_2020_2024.csv', index=False)

In [ ]:
Clean the Population data

In [ ]:
import pandas as pd
import re
import os
import glob

def clean_census_population_auto(file_path):
    # 1. Extract the Year from the filename (e.g., 2021)
    match = re.search(r'20\d{2}', os.path.basename(file_path))
    year = int(match.group(0)) if match else None

    # 2. Load the file
    df = pd.read_csv(file_path)

    # 3. Extract State names
    # In these files, state names are rows that DON'T contain 'Estimate' or 'Percent'
    df['State'] = df['Label (Grouping)'].apply(
        lambda x: x.strip() if 'Estimate' not in str(x) and 'Percent' not in str(x) else None
    )
    # Forward-fill: This copies the State name down to the 'Estimate' row below it
    df['State'] = df['State'].ffill()

    # 4. Filter for only the 'Estimate' rows
    df_estimates = df[df['Label (Grouping)'].str.contains('Estimate', na=False)].copy()

    # 5. Define target columns (Total Population and 18+ for voting analysis)
    target_cols = {
        'SEX AND AGE!!Total population': 'Total_Population',
        'SEX AND AGE!!Total population!!18 years and over': 'Voter_Age_Population'
    }

    # Rename columns and add the Year
    df_estimates = df_estimates.rename(columns=target_cols)
    df_estimates['Year'] = year

    # Keep only the columns we need
    result_df = df_estimates[['State', 'Year', 'Total_Population', 'Voter_Age_Population']].copy()

    # 6. Clean numeric strings (remove commas and Census symbols like '-' or 'N')
    def clean_numeric(val):
        if pd.isna(val) or str(val).strip() in ['-', 'N', '(X)', '']:
            return None
        try:
            return int(float(str(val).replace(',', '').strip()))
        except (ValueError, TypeError):
            return None

    result_df['Total_Population'] = result_df['Total_Population'].apply(clean_numeric)
    result_df['Voter_Age_Population'] = result_df['Voter_Age_Population'].apply(clean_numeric)

    return result_df

# --- COMBINE ALL YEARS ---
# Using a pattern to find your uploaded files
all_pop_files = glob.glob('population_20*.csv')
all_dfs = []

for f in all_pop_files:
    clean_df = clean_census_population_auto(f)
    if clean_df is not None:
        all_dfs.append(clean_df)

if all_dfs:
    # Combine everything and filter for the 2020-2024 range
    population_panel = pd.concat(all_dfs, ignore_index=True)
    population_panel = population_panel[population_panel['Year'].between(2020, 2024)]

    # Save the final file for Tableau
    population_panel.to_csv('population_panel_2020_2024.csv', index=False)
    print("Success! Your 2020-2024 population panel is ready.")

Success! Your 2020-2024 population panel is ready.
